# 11_00 — Export the offline Kaggle inference bundle

**CLOSED-WORLD / CLASS-1 ONLY.** Candidates are the training structure library. There is no
external candidate expansion and no Mode B. Submission 1 is an anchor, not the final solution.

This notebook writes `bundle/`: structures, adducts, ppm windows, reference metadata, CSR index,
peak store, models, config, code, README and a sha256 manifest. It can run **before** the
scaling track finishes. It uses the v5 frozen spectrum model if
`outputs/v5/freeze/spectrum_model.json` exists, and otherwise the historical v4b V1. Swapping the
model later only re-runs section 6 and the manifest refresh.

The reference-library export streams all ~2.5M train spectra once (peak cleaning, deterministic
top-100 and T1 peak hash). That is the only slow step, and it is skipped when an export with
the same inputs already exists.

In [1]:
import sys, json, hashlib
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'src').exists() and (REPO_ROOT.parent / 'src').exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))

import numpy as np
import pandas as pd

from casmi.paths import get_project_paths
from casmi.io.artifacts import load_artifact
from casmi.candidates.generator import CandidateGenerationConfig
from casmi.chemistry.adducts import neutral_mass_from_precursor
from casmi.qcr.context import v4b_paths, v5_paths
from casmi.qcr.fingerprint import code_version
from casmi.qcr.settlement import load_settlement_or_refuse
from casmi.ranking.lambdamart import BASE_FEATURES
from casmi import bundle_export as bx
from casmi_infer.adducts import AdductRules
from casmi_infer.validation import check_bundle_invariants, sha256_file, verify_bundle

paths = get_project_paths()
vp, D = v4b_paths(paths), v5_paths(paths)
BUNDLE = D['bundle']
BUNDLE.mkdir(parents=True, exist_ok=True)
REBUILD_REFERENCE_LIBRARY = False     # True forces the ~2.5M-spectrum peak-store re-export
COMPUTE_PEAK_HASH = True              # T1 exact-mirror check at inference needs it (slowest part of the export)
settlement = load_settlement_or_refuse(vp.settlement_json)
SIM_CFG = settlement['similarity_config']

freeze_path = D['freeze'] / 'spectrum_model.json'
if freeze_path.exists():
    freeze = json.loads(freeze_path.read_text(encoding='utf-8'))
    MODEL_DIR, MODEL_ID, FEATURE_NAMES = Path(freeze['model_dir']), freeze['model_id'], freeze['feature_names']
    MODEL_EXTRA = {'freeze_status': freeze.get('freeze_status'), 'spectrum_model_json_sha256': sha256_file(freeze_path)}
else:
    MODEL_DIR, MODEL_ID = vp.out / 'models' / 'V1', 'V1_v4b_DEV1k'
    FEATURE_NAMES = json.loads((MODEL_DIR / 'model_meta.json').read_text(encoding='utf-8'))['feature_cols']
    MODEL_EXTRA = {'freeze_status': 'HISTORICAL v4b V1 (pre-scaling anchor)'}
assert list(FEATURE_NAMES) == list(BASE_FEATURES), 'frozen feature order differs from the V1 recipe'
print('model to export:', MODEL_ID, '<-', MODEL_DIR)

model to export: V1_TL_1K <- C:\Users\myben\OneDrive\Documents\Enveda\outputs\v5\models\V1_TL_1K


## 1. Structures: the exact training candidate library (mass variants, sorted by float64 mass)

In [2]:
mmv = pd.read_parquet(paths.interim / 'candidate_generation' / 'molecule_mass_variants.parquet')
structures, conn = bx.export_structures(mmv, BUNDLE)
print(f'{len(structures):,} mass-variant rows, {len(conn):,} connectivities')

274,330 mass-variant rows, 274,195 connectivities


## 2. Adducts (serialized from the training parser) + parity against it

In [3]:
adduct_ref = pd.read_parquet(paths.processed / 'adduct_reference.parquet')
test_meta = pd.read_parquet(paths.processed / 'test_spectrum_metadata.parquet')
adducts_payload = bx.export_adducts(adduct_ref['adduct'].astype(str), test_meta['adduct'].astype(str).unique(), BUNDLE)
rules = AdductRules.load(BUNDLE / 'adducts.json')
mz_grid = np.linspace(80.0, 2000.0, 97)
parity_bad = [(a, mz) for a in rules.rules for mz in mz_grid
              if (rules.neutral_mass(mz, a) is None) != (neutral_mass_from_precursor(mz, a) is None)
              or (rules.neutral_mass(mz, a) is not None and rules.neutral_mass(mz, a) != neutral_mass_from_precursor(mz, a))]
print(f'adduct rules: {len(rules.rules)} | unsupported observed test adducts: {adducts_payload["unsupported_observed_test_adducts"]}')
print('adduct export parity violations (exact equality required):', len(parity_bad))
assert not parity_bad

adduct rules: 121 | unsupported observed test adducts: []
adduct export parity violations (exact equality required): 0


## 3. Retrieval windows (from the training contract)

In [4]:
with open(paths.processed / 'candidate_generation' / 'candidate_generation_contract.json') as f:
    contract = json.load(f)
ppm_payload = bx.export_ppm_windows(contract, adducts_payload, BUNDLE, CandidateGenerationConfig())
print({k: v for k, v in ppm_payload.items() if k != 'per_adduct_primary_ppm'})

{'source': 'candidate_generation_contract.json molecule_policy.tolerance_ppm (global; the adaptive per-group policy was evaluated in notebook 03 but not adopted by the contract)', 'default_primary_ppm': 50.0, 'max_standard_ppm': 50.0, 'fallback_ppm': [100.0, 200.0], 'nearest_n': 25, 'fallback_sequence': ['primary', 'ppm_100', 'ppm_200', 'nearest_25'], 'fallback_note': 'fallback levels are INFERENCE-ONLY (training never needed them: every training query had a non-empty primary window by construction); every use is logged'}


## 4. Reference library: metadata, CSR index, similarity-representation peak store

In [5]:
ref_files = ['ref_meta.parquet', 'ref_index_offsets.npy', 'ref_index_ids.npy', 'ref_peaks_offsets.npy', 'ref_peaks_mz.npy', 'ref_peaks_int.npy']
train_meta = load_artifact('train_spectrum_metadata')
lib_inputs = {'train.parquet': f'{paths.train.stat().st_size}|{paths.train.stat().st_mtime:.0f}', 'max_peaks': SIM_CFG['max_peaks_similarity'],
              'n_train_rows': len(train_meta), 'peak_hash': COMPUTE_PEAK_HASH, 'connectivity_table_sha256': sha256_file(BUNDLE / 'connectivities.parquet')}
stamp = BUNDLE / '_reference_library_inputs.json'
reuse = (not REBUILD_REFERENCE_LIBRARY and stamp.exists() and all((BUNDLE / f).exists() for f in ref_files)
         and json.loads(stamp.read_text(encoding='utf-8')) == lib_inputs)
if reuse:
    print('reference library: reusing the existing export (inputs unchanged)')
else:
    info = bx.export_reference_library(train_meta, paths.train, conn, BUNDLE, max_peaks=SIM_CFG['max_peaks_similarity'], compute_hash=COMPUTE_PEAK_HASH)
    stamp.write_text(json.dumps(lib_inputs, indent=2), encoding='utf-8')
    print(info)
problems = check_bundle_invariants(structures, conn, np.load(BUNDLE / 'ref_index_offsets.npy'), np.load(BUNDLE / 'ref_peaks_offsets.npy'), len(train_meta))
print('bundle invariants:', problems or 'OK')
assert not problems

[export] 250,000/2,539,608 reference spectra
[export] 500,000/2,539,608 reference spectra
[export] 750,000/2,539,608 reference spectra
[export] 1,000,000/2,539,608 reference spectra
[export] 1,250,000/2,539,608 reference spectra
[export] 1,500,000/2,539,608 reference spectra
[export] 1,750,000/2,539,608 reference spectra
[export] 2,000,000/2,539,608 reference spectra
[export] 2,250,000/2,539,608 reference spectra
[export] 2,500,000/2,539,608 reference spectra
{'n_refs': 2539608, 'n_peaks': 132580562, 'n_refs_without_library_connectivity': 0}
bundle invariants: OK


## 5. Peak-store spot check: exported arrays == training similarity representation

In [6]:
from casmi.io.parquet import load_rows_by_offset
from casmi.spectra.preprocessing import remove_invalid_peaks, truncate_top_peaks
from casmi_infer.reference_index import ReferenceLibrary
lib = ReferenceLibrary(BUNDLE)
rng = np.random.default_rng(42)
sample_rows = np.sort(rng.choice(len(train_meta), size=500, replace=False))
raw = load_rows_by_offset(paths.train, ['ms2_mzs', 'ms2_normalized_intensities', 'precursor_mz'], sample_rows).rename(columns={'_row_offset': 'row_offset'})
bad = 0
for r in raw.itertuples(index=False):
    mz, it = remove_invalid_peaks(r.ms2_mzs, r.ms2_normalized_intensities)
    s_mz, s_it = truncate_top_peaks(mz, it, max_peaks=SIM_CFG['max_peaks_similarity'])
    p = lib.peaks(int(r.row_offset))
    bad += int(not (np.array_equal(p['mzs'], s_mz) and np.array_equal(p['intensities'], s_it) and p['precursor_mz'] == float(r.precursor_mz)))
print('peak-store spot-check mismatches (500 random references):', bad)
assert bad == 0

peak-store spot-check mismatches (500 random references): 0


## 6. Models (swap point) + config + code + README + manifest

In [7]:
model_info = bx.export_models(MODEL_DIR, BUNDLE, MODEL_ID, FEATURE_NAMES, tag='v1', extra_info=MODEL_EXTRA)
config = bx.build_config(SIM_CFG)
(BUNDLE / 'config.json').write_text(json.dumps(config, indent=2), encoding='utf-8')
code_files = bx.export_code(REPO_ROOT / 'src', BUNDLE)
(BUNDLE / 'README.md').write_text(bx.README, encoding='utf-8')
source_artifacts = {
    'molecule_mass_variants.parquet': sha256_file(paths.interim / 'candidate_generation' / 'molecule_mass_variants.parquet'),
    'adduct_reference.parquet': sha256_file(paths.processed / 'adduct_reference.parquet'),
    'candidate_generation_contract.json': sha256_file(paths.processed / 'candidate_generation' / 'candidate_generation_contract.json'),
    'train_spectrum_metadata.parquet': sha256_file(paths.processed / 'train_spectrum_metadata.parquet'),
    'train.parquet (size|mtime)': lib_inputs['train.parquet'],
    'cache_settlement.evidence_fingerprint': settlement['evidence_fingerprint'],
    'model_files': model_info['file_sha256'],
}
manifest = bx.refresh_manifest(BUNDLE, source_artifacts=source_artifacts, code_version=code_version(),
                               extra={'model_id': MODEL_ID, 'bundle_version': f"{config['CONFIG_HASH']}-{MODEL_ID}"})
print('CONFIG_HASH', config['CONFIG_HASH'], '| files:', len(manifest['files']), '| code files:', len(code_files))

CONFIG_HASH 128372cae78ad88b | files: 47 | code files: 26


## 7. Verify (sha256 of every file + CONFIG_HASH) and size report

In [8]:
_, _, problems = verify_bundle(BUNDLE)
print('bundle verification:', problems or 'OK')
assert not problems
sizes = pd.DataFrame([{'file': k, 'MB': (BUNDLE / k).stat().st_size / 1e6} for k in manifest['files']]).sort_values('MB', ascending=False)
display(sizes.head(15)); print(f"total {sizes['MB'].sum() / 1e3:.2f} GB")
print('Upload bundle/ MANUALLY as a private Kaggle dataset; nothing here uploads anything.')

bundle verification: OK


,file,MB
43,ref_peaks_mz.npy,1060.644624
42,ref_peaks_int.npy,1060.644624
41,ref_meta.parquet,224.623066
44,ref_peaks_offsets.npy,20.317000
39,ref_index_ids.npy,20.316992
45,structures.parquet,17.276176
29,connectivities.parquet,12.446093
46,structures_mass.npy,2.194768
40,ref_index_offsets.npy,2.193696
33,models/v1_fold1.txt,1.073944


total 2.43 GB
Upload bundle/ MANUALLY as a private Kaggle dataset; nothing here uploads anything.
